In [16]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# READ I/O

In [17]:
from pathlib import Path

import numpy as np
import os
import pickle
import pandas as pd
import sys
import networkx as nx
import matplotlib.pyplot as plt
import json

# import multidict

from itertools import combinations
from collections import OrderedDict

In [18]:
# Get the current working directory
cwd = Path.cwd()

# Get the path to the parent directory
parent_dir = cwd.parent

# Add the parent directory to the Python path
sys.path.append(str(parent_dir))

In [19]:

folder_path = '/Users/gl665/Documents/Phrame-MentalMaps-1/data/jsonfiles'
count = 1
clds_dict = OrderedDict()
for filename in os.listdir(folder_path):
    if filename.endswith('.json'):
        file_path = os.path.join(folder_path, filename)
        with open(file_path, 'rb') as f:
            print(file_path)
            data = pickle.load(f)  # list

            # convert the list to a nx graph.
            G = nx.from_dict_of_dicts(data[1], create_using=nx.DiGraph)
            G = nx.relabel_nodes(G, data[0])

            clds_dict[count] = {
                'name': filename,
                'cld_graph': G
            }
            count += 1

# print(clds_dict.keys(), clds_dict.values())

/Users/gl665/Documents/Phrame-MentalMaps-1/data/jsonfiles/E clean model 572020_edited.json
/Users/gl665/Documents/Phrame-MentalMaps-1/data/jsonfiles/C 7 6 20 clean model_edited.json
/Users/gl665/Documents/Phrame-MentalMaps-1/data/jsonfiles/DSS full clean_edited.json
/Users/gl665/Documents/Phrame-MentalMaps-1/data/jsonfiles/j model 7 13 20 clean model_edited.json
/Users/gl665/Documents/Phrame-MentalMaps-1/data/jsonfiles/full circle GBM 5 24 21clean_edited.json
/Users/gl665/Documents/Phrame-MentalMaps-1/data/jsonfiles/Passages Clean_edited.json
/Users/gl665/Documents/Phrame-MentalMaps-1/data/jsonfiles/Wavi combined clean_edited.json
/Users/gl665/Documents/Phrame-MentalMaps-1/data/jsonfiles/reentry clean model_revised.json
/Users/gl665/Documents/Phrame-MentalMaps-1/data/jsonfiles/nat model clean_edited2.json
/Users/gl665/Documents/Phrame-MentalMaps-1/data/jsonfiles/FC Prov Clean_edited.json
/Users/gl665/Documents/Phrame-MentalMaps-1/data/jsonfiles/4 28 2021 modelclean_edited.json
/Users/g

In [20]:
input_file = 'data/data_dictionary/RC_models_data dictionary.xlsx'


cwd = os.getcwd()

file_path = str(parent_dir / input_file)

# Read all sheets into a dictionary of DataFrames
excel_data = pd.read_excel(file_path, sheet_name=None)

data_dictionary = {}
# key is the sheet name, value is the DataFrame

for k, v in excel_data.items():
    # rename columns header names
    v.columns.values[0] = "variable"
    v.columns.values[1] = "definition"

    # v = v.to_dict(orient="records")  # a list of dictionaries
    k = int(k)

    data_dictionary[k] = v

In [21]:
from sentence_transformers import SentenceTransformer


def find_node_name_matches(vensim_nodes, vars_def_pair):

    model = SentenceTransformer("all-mpnet-base-v2")

    embeddings_cld = model.encode(vensim_nodes)
    excel_nodes = list(vars_def_pair.keys())
    # print(excel_nodes)

    embeddings_excel = model.encode(excel_nodes)

    node_var_def_dict = {}

    similarities = model.similarity(embeddings_cld, embeddings_excel)
    # Output the pairs with their score
    for idx_i, vensim_node in enumerate(vensim_nodes):

        matches = []

        for idx_j, excel_node in enumerate(excel_nodes):
            if (similarities[idx_i][idx_j] > 0.7):
                # match vensim and excel nodes when similarity > 0.7

                # print(
                #     f"{vensim_node} ---- {excel_node}: {similarities[idx_i][idx_j]:.2f}")
                matches.append([excel_node, vars_def_pair[excel_node]])

        if matches:
            node_var_def_dict[vensim_node] = matches

    return node_var_def_dict

In [22]:

# get the data dictionary with both the graph strucutre and the variable definitions

all_vars_def = {}

for v, k in clds_dict.items():
    # print(v)

    nx_graph = clds_dict[v]["cld_graph"]
    node_names = list(nx_graph.nodes)
    # print(node_names[10:])

    df = data_dictionary[v]
    # print(df.head())

    vars_def_pair = dict(zip(df.iloc[:, 0], df.iloc[:, 1]))

    node_var_def_dict = find_node_name_matches(node_names, vars_def_pair)
    clds_dict[v]["node_var_def_dict"] = node_var_def_dict

    all_vars_def.update(node_var_def_dict)

In [23]:
# with open('all_vars_def_dictionary.pkl', 'wb') as f:
#     pickle.dump(all_vars_def, f)


# with open("/Users/gl665/Documents/Phrame-MentalMaps-1/notebook/my_dict.pkl", "rb") as f:
#     cld_data_dictionary = pickle.load(f)

In [24]:
cld_data_df = pd.DataFrame.from_dict(
    all_vars_def, orient="index").reset_index()

print("total number of rows: ", len(cld_data_df))

cld_data_df.head()

total number of rows:  1074


,index,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18
0,alcohol misuse,"[alcohol misuse, using alcohol - in reference ...",None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
1,intimate partner violence,"[intimate partner violence, perpetration or vi...",None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2,mental health,"[mental health, ""Increases"" in mental health i...",None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
3,reportable incident to police,"[reportable incident to police, the occurrence...",None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
4,reporting incident of intimate partner violence,"[intimate partner violence, perpetration and o...",[reporting incident of intimate partner violen...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None


In [25]:
all_vars_list = cld_data_df['index'].tolist()


print(all_vars_list)


def create_all_pairs(variables):
    # Generate all possible pairs
    all_pairs = list(combinations(variables, 2))
    return all_pairs


pairs = create_all_pairs(all_vars_list)
# print(pairs)

# make it into pandas dataframe
df_pairs = pd.DataFrame(pairs, columns=["sentence1", "sentence2"])

print("total number of pairs: ", len(df_pairs))
print(df_pairs.head())

# export to csv
df_pairs.to_csv("all_variable_pairs.csv", index=False)

['alcohol misuse', 'intimate partner violence', 'mental health', 'reportable incident to police', 'reporting incident of intimate partner violence', 'institutional racism against indigenous people', 'police involvement', 'quality of treatment by police', 'percieved victimization stigma', 'trust in police', 'dependence on partner', 'reluctance to report', 'ability to be self sustaining', 'financial need', 'housing need', 'availability of financial resources', 'seeking financial resources', 'recieving financial resources', 'recieving housing resources', 'seeking housing resources', 'availability of housing resources', 'normalization of violence', 'children witnessing intimate partner violence', 'historical trauma', 'childhood home functioning', 'adverse childhood experiences', 'fear of police', "indigenous peoples' experienced discrimination by police", 'police violence against indigenous people', 'police deescalation training', 'police antiracist training', 'victim ends relationship wit

In [26]:
definition_list = []

for k, v in all_vars_def.items():
    for item in v:
        definition_list.append(item[1])

In [27]:

pairs = create_all_pairs(definition_list)
# print(pairs)

# make it into pandas dataframe
df_pairs = pd.DataFrame(pairs, columns=["sentence1", "sentence2"])

print("total number of pairs: ", len(df_pairs))
print(df_pairs.head())

# export to csv
df_pairs.to_csv("all_definition_pairs.csv", index=False)

total number of pairs:  2299440
                                sentence1  \
0  using alcohol - in reference to misuse   
1  using alcohol - in reference to misuse   
2  using alcohol - in reference to misuse   
3  using alcohol - in reference to misuse   
4  using alcohol - in reference to misuse   

                                           sentence2  
0  perpetration or victimization of violence by a...  
1  "Increases" in mental health indicate a declin...  
2  the occurrence of a (physical mostly) violent ...  
3  perpetration and or victimization of violence ...  
4               reporting an IPV event to the police  
